### Background

This Jupyter Notebook demonstrates how we generated additional data to expand our training set. The main goal of data generation was to address class imbalance and enhance the diversity of our dataset, ultimately improving the performance of our model. We used the OpenAI API as the model provider for data generation.

Before start, you need to install the following dependencies:

```
python-dotenv
openai
pandas
numpy
tqdm
```

Commanda if you use conda environment:

```
conda activate <env-name>
conda install pandas numpy -y
pip install python-dotenv
pip install openai
conda install conda-forge::tqdm
```

Create `.env` file with `OPENAI_API_KEY`

### Imports

In [19]:
import os

import numpy as np
import pandas as pd

from tqdm import tqdm
from dotenv import load_dotenv

from openai import OpenAI

### Init OpenAI client

In [20]:
load_dotenv()

True

In [21]:
client = OpenAI()

### Constants

In [22]:
TRAIN_PATH = "data/"
TRAIN_NAME = "train.parquet"

### Read Data

In [23]:
df = pd.read_parquet(os.path.join(TRAIN_PATH, TRAIN_NAME))
df.shape

(3822, 6)

In [24]:
df.head()

,id,content,lang,manipulative,techniques,trigger_words
0,0bb0c7fa-101b-4583-a5f9-9d503339141c,Новий огляд мапи DeepState від російського вій...,uk,True,"[euphoria, loaded_language]","[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,Недавно 95 квартал жёстко поглумился над русск...,ru,True,"[loaded_language, cherry_picking]","[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,e6a427f1-211f-405f-bd8b-70798458d656,🤩\nТим часом йде евакуація Бєлгородського авто...,uk,True,"[loaded_language, euphoria]","[[55, 100]]"
3,1647a352-4cd3-40f6-bfa1-d87d42e34eea,В Україні найближчим часом мають намір посилит...,uk,False,None,None
4,9c01de00-841f-4b50-9407-104e9ffb03bf,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",ru,True,[loaded_language],"[[114, 144]]"


In [25]:
df = df.loc[~df["techniques"].isna()]
df.shape

(2589, 6)

In [26]:
# Explode and count
label_counts = df["techniques"].explode().value_counts()
print(label_counts)

techniques
loaded_language            1973
cherry_picking              512
glittering_generalities     483
cliche                      463
euphoria                    462
fud                         385
appeal_to_fear              300
whataboutism                158
bandwagon                   157
straw_man                   138
Name: count, dtype: int64


In [27]:
labels_to_keep = {
    "straw_man", 
    "bandwagon", 
    "whataboutism", 
    "appeal_to_fear", 
    "fud",
    # "euphoria",
    # "cliche",
    # "glittering_generalities",
    # "cherry_picking",
    # "loaded_language",
}
filtered_df = df[df["techniques"].apply(lambda x: any(label in labels_to_keep for label in x))]
print(filtered_df.shape)

filtered_df["techniques"].explode().value_counts()

(903, 6)


techniques
loaded_language            684
fud                        385
appeal_to_fear             300
cherry_picking             279
cliche                     188
whataboutism               158
bandwagon                  157
straw_man                  138
glittering_generalities     86
euphoria                    46
Name: count, dtype: int64

### Helpers

In [28]:
def extract_phrases(content, trigger_words):
    """
    Extracts phrases from the content based on trigger word indices.

    :param content: The input text.
    :param trigger_words: List of index ranges [[start1, end1], [start2, end2], ...].
    :return: List of extracted phrases.
    """
    return [content[start:end] for start, end in trigger_words]

In [29]:
def check_spans_in_text(text: str, spans: list[str]) -> tuple[dict[str, bool], float]:
    """
    Check if each span in the list is a substring of the given text.
    """
    results = {span: span in text for span in spans}
    true_count = sum(results.values())  # Count how many spans are found
    success_rate = true_count / len(spans) if spans else 0.0  # Avoid division by zero
    
    return results, success_rate

# Example usage:
text_sample = "Artificial intelligence is transforming the world."
spans_list = ["Artificial intelligence", "Machine learning", "transforming"]

result_dict, rate = check_spans_in_text(text_sample, spans_list)

round(rate, 2), result_dict

(0.67,
 {'Artificial intelligence': True,
  'Machine learning': False,
  'transforming': True})

In [30]:
def find_spans_in_text(text: str, spans: list[str]) -> list[list[int]]:
    """
    Find the start and end indices of spans within the text.
    """
    found_spans = []
    
    for span in spans:
        start_idx = text.find(span)  # Find first occurrence
        if start_idx != -1:
            end_idx = start_idx + len(span)  # Compute end index
            found_spans.append([start_idx, end_idx])
    
    return found_spans

### Prompting

In [31]:
def generate_prompt(text: str, triggers: list[str]):
    prompt = f"""You are telegram editor. You need to create a unique samples based on the given text.

Rephrase or generate a similar text to the given text [TEXT].
IMPORTANT: Leave the phrases defined in list [UNCHANGED] exactly the same (do not highlight them please).

Be creative so the text is not very similar to the given one.
Preserve the meaning and the style of the text. Also, make sure you are using the same language and style as the original text.

[TEXT]
{text}

[UNCHANGED]
{triggers}"""
    return prompt

In [32]:
def generate_sample(prompt: str):
    completion = client.chat.completions.create(
        model="gpt-4o",
        store=True,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    return completion.choices[0].message.content

In [33]:
def generate_sample(prompt: str):
    completion = client.chat.completions.create(
        model="gpt-4o",
        store=True,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    return completion.choices[0].message.content

In [34]:
example = df.sample(1)

techniques = example["techniques"].iloc[0]
text = example["content"].iloc[0]
triggers = extract_phrases(text, example["trigger_words"].iloc[0])
print(example["trigger_words"].iloc[0])
prompt = generate_prompt(text, triggers)

print(techniques)
print("-" * 50)
print(text)
print("-" * 50)
print(triggers)
print("-" * 50)
print(prompt)

[array([169, 208]) array([321, 334]) array([338, 352]) array([364, 394])
 array([431, 442]) array([447, 454]) array([456, 463]) array([492, 501])
 array([338, 341]) array([533, 548])]
['loaded_language' 'cherry_picking' 'fud']
--------------------------------------------------
ВЕЧірній Зеленський:
"Через терористичний напад на Ізраїль та загрозу нашим громадянам, які перебувають в Ізраїлі, створено оперативний штаб при МЗС України."
Lida Sha:
Так би за українців в Україні переживав  - "зберігайте спокій, залишайтесь вдома, ми працюємо" звернення 24.02.22
Iryna:
А знаєте, чому весь світ тепер буде прикутий до жахливих відео в Їзраїлі?
Бо в Україні політика мовчання.
Я давно про це кажу, там, де влада має горлати про жахіття, геноцид, вона мовчить... 
Ми й самі прозріємо про той жах, коли оприлюднять.
Але це не точно...
--------------------------------------------------
['Так би за українців в Україні переживав', 'буде прикутий', 'жахливих відео', 'Бо в Україні політика мовчання', 'має г

In [35]:
%%time

generated_sample = generate_sample(prompt)
print(generated_sample)
print()

Зелена хрибустина Володимира:  
"У зв'язку з терористичним нападом у Ізраїлі та загрозами нашим громадянам там, Міністерством закордонних справ України створено кризовий центр."

Lida Sha:  
Так би за українців в Україні переживав - "зберігайте спокій, залишайтесь вдома, ми працюємо" звернення 24.02.22

Iryna:  
А знаєте, чому весь світ тепер буде прикутий до жахливих відео в Їзраїлі?  
Бо в Україні політика мовчання.  
Я давно підозрюю, там, де влада має горлати про жахіття, геноцид, вона мовчить...  
Ми й самі прозріємо про той жах, коли оприлюднять.  
Але це не точно...

CPU times: user 12.2 ms, sys: 9.4 ms, total: 21.6 ms
Wall time: 6.23 s


In [36]:
res_dict, res_rate = check_spans_in_text(generated_sample, triggers)

print(round(res_rate, 2))
print(res_dict)

1.0
{'Так би за українців в Україні переживав': True, 'буде прикутий': True, 'жахливих відео': True, 'Бо в Україні політика мовчання': True, 'має горлати': True, 'жахіття': True, 'геноцид': True, 'прозріємо': True, 'жах': True, 'Але це не точно': True}


In [37]:
find_spans_in_text(generated_sample, triggers)

[[191, 230],
 [345, 358],
 [362, 376],
 [390, 420],
 [456, 467],
 [472, 479],
 [481, 488],
 [518, 527],
 [362, 365],
 [561, 576]]

### Generation

In [38]:
df.head()

,id,content,lang,manipulative,techniques,trigger_words
0,0bb0c7fa-101b-4583-a5f9-9d503339141c,Новий огляд мапи DeepState від російського вій...,uk,True,"[euphoria, loaded_language]","[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,Недавно 95 квартал жёстко поглумился над русск...,ru,True,"[loaded_language, cherry_picking]","[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,e6a427f1-211f-405f-bd8b-70798458d656,🤩\nТим часом йде евакуація Бєлгородського авто...,uk,True,"[loaded_language, euphoria]","[[55, 100]]"
4,9c01de00-841f-4b50-9407-104e9ffb03bf,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",ru,True,[loaded_language],"[[114, 144]]"
5,46493f44-f00a-4ffb-9cda-252ccf5fa4c6,"Апартаменти\n триповерхова келія Паші Лєбєдя, ...",uk,True,[loaded_language],"[[94, 108], [208, 227]]"


In [39]:
# data = df.sample(20)
data = filtered_df

result = []
for index, row in tqdm(data.iterrows(), total=len(data)):
    try:
        triggers = extract_phrases(row["content"], row["trigger_words"])
        prompt = generate_prompt(row["content"], triggers)
        
        generated_content = generate_sample(prompt)
        # generated_content = ""
        generation_checker, generation_rate = check_spans_in_text(generated_content, triggers)

        generated_trigger_words = find_spans_in_text(generated_content, triggers)
    
        result.append({
            "id": row["id"], # id of original sample
            "prompt": prompt,
            "generated_content": generated_content,
            "trigger_words_string": triggers,
            "generated_trigger_words": generated_trigger_words,
            "generation_rate": round(generation_rate, 2),
            "generation_checker": generation_checker,
        })
    except e:
        print(e)

100%|█████████████████████████████████████████████| 903/903 [1:30:41<00:00,  6.03s/it]


In [40]:
#  id  - id of original sample
#  prompt - used prompt
#  generated_content - generated content
#  trigger_words_string - list of spans that need to be without changes
#  generated_trigger_words - span location in generated text
#  generation_rate - quality of generated text: if you can match only 2 of 3 spans, rate is 2/3 ~ 0.67
#  generation_checker - dict of true/false of detected spans for debuging

generated_df = pd.DataFrame(result)
generated_df.head(10)

,id,prompt,generated_content,trigger_words_string,generated_trigger_words,generation_rate,generation_checker
0,7d7504ff-295e-435d-b565-ed62d4eebaa0,You are telegram editor. You need to create a ...,С 1994 года Российская Федерация ведет боевые ...,[Российское руководство сделало террор инструм...,"[[935, 950], [1094, 1110], [1144, 1183], [1300...",0.67,{'Российское руководство сделало террор инстру...
1,69912f3c-29e5-40ac-a12d-0226e4971615,You are telegram editor. You need to create a ...,"Ну, что, расеянцы, поборолись с Украиной? А ес...","[Ну, что, расеянцы, повоевали, А если бы учили...","[[0, 17], [42, 65], [87, 112], [163, 182]]",0.80,"{'Ну, что, расеянцы': True, 'повоевали': False..."
2,c598905d-07f5-4bf0-9292-fb83111a4294,You are telegram editor. You need to create a ...,"Очевидні речі, але ще є ті, для кого це не так...","[сукупні ресурси антивоєнної, демократичної, п...","[[126, 153], [155, 168], [170, 184], [186, 223...",0.88,"{'сукупні ресурси антивоєнної': True, 'демокра..."
3,19daeb3a-51ab-4899-811c-81b3460fc4ef,You are telegram editor. You need to create a ...,❗️ \nНа розі вулиці Хрещатик та Бессарабської...,[На розі вулиці Хрещатик та Бессарабської площ...,"[[5, 96], [179, 215], [223, 286], [294, 337], ...",0.90,{'На розі вулиці Хрещатик та Бессарабської пло...
4,1b9e8836-707c-4a8b-b093-7ccc11bdb10e,You are telegram editor. You need to create a ...,Зупиніться з нищенням екосистеми України безжа...,"[безжально знищувати екосистему, безглуздого с...","[[228, 259], [335, 353], [457, 480]]",0.75,"{'безжально знищувати екосистему': False, 'без..."
5,a7979f8a-c586-4d72-aae2-1d80323b7a0e,You are telegram editor. You need to create a ...,"🇺🇦 \nУкраине нужны солдаты, а не учёные \n🤦 ...","[Украине нужны солдаты, а не учёные, ВУЗы Льво...","[[5, 39], [353, 432]]",0.67,"{'Украине нужны солдаты, а не учёные': True, '..."
6,87270541-c2b4-4b41-b411-e75334bcc63b,You are telegram editor. You need to create a ...,⚡️\n\nПрямо зараз світ знаходиться на межі тре...,[Прямо зараз світ знаходиться на межі третьої ...,"[[186, 295]]",0.50,{'Прямо зараз світ знаходиться на межі третьої...
7,dcef63b7-de77-4394-940e-707d3b2f0761,You are telegram editor. You need to create a ...,Шановне товариство!!! Рекомендую ознайомитися ...,[Запрошую на YouTube-канал \n«Орестократія». В...,"[[654, 806]]",1.00,{'Запрошую на YouTube-канал «Орестократія». В...
8,ff2e821f-d786-4c47-8daf-f69ee337c714,You are telegram editor. You need to create a ...,Журнал «Forbes» выявил ещё одну причину провал...,"[провала контрнаступа, всушники, контрнаступ з...","[[40, 60], [179, 187], [228, 251]]",0.75,"{'провала контрнаступа': True, 'всушники': Tru..."
9,bb926728-461d-4fc0-8cc5-206245492e0d,You are telegram editor. You need to create a ...,Вступление\n\n🔼\n🔼\n🔼\nНевольно вспоминается л...,[«оранжевую революцию» еще за два года до ее н...,"[[842, 927], [929, 1029], [1366, 1419], [1596,...",0.67,{'«оранжевую революцию» еще за два года до ее ...


In [48]:
generated_df.to_csv("data/generated_sample.csv", index=False)

In [49]:
generated_df["generation_rate"].mean()

np.float64(0.689014396456257)

In [50]:
generated_df.shape

(903, 7)

In [51]:
generated_df.loc[generated_df["generation_rate"] == 1].to_csv("data/qualitative_generated_sample.csv", index=False)

In [52]:
generated_df.loc[generated_df["generation_rate"] == 1].shape

(363, 7)

In [43]:
# de-bugging
# for _, row in generated_df.iterrows():
#     print(row["generated_content"])